# Lab 3 · Score it

**DemoWallet risk lab** · Securing AI-Driven Mobile Experiences · DSC Europe

The server in Lab 2 already turns device evidence into a decision: **allow · monitor · step_up · block**.
In this notebook you add the missing layer, **behavior**, then tune the thresholds and check *who* you end up challenging.

* **Step 1 (slide 25, ~10 min):** build a score, choose thresholds, write down three numbers.
* **Step 2 (slide 26, ~5 min):** look at false positives per cohort, then fix what you find.

Run the cells top to bottom (**Runtime → Run all** works too). Cells marked **✏️ TODO** are yours to edit; each one runs as-is, so you can't break the notebook.
Stuck? Open `risk_lab_solution.ipynb`.

> **The data is synthetic.** 6,200 transfer attempts generated by a seeded script (`lab3/generate_sessions.py`). Each row is labelled *human*, *legitimate_agent* or *bot*: the question is always **“human or not”, never “which human”**. There are no user identifiers and no content, only derived timing and shape features.

In [ ]:
# @title Setup (numpy, pandas, matplotlib — all preinstalled in Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

In [ ]:
# @title The dataset generator (identical copy of lab3/generate_sessions.py) { display-mode: "form" }
# Building the data here means Colab needs no download: it works on conference Wi-Fi
# and doesn't depend on the repo URL. The fingerprint check below proves it's the same table.
import hashlib

import numpy as np
import pandas as pd

SEED = 2026
COUNTS = {"human": 4300, "screen_reader": 300, "legitimate_agent": 1000, "bot": 600}
BOT_KINDS = {"script": 210, "a11y_agent": 240, "farm": 150}


def make_sessions(seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    def pick(n, options, p):
        return rng.choice(np.array(options, dtype=object), size=n, p=p)

    def flag(n, p):
        return (rng.random(n) < p).astype(int)

    def lognorm(n, median, sigma):
        return np.exp(rng.normal(np.log(median), sigma, n))

    def beta(n, a, b):
        return rng.beta(a, b, n)

    def block(cohort, n, kind=""):
        d = {"cohort": [cohort] * n, "bot_kind": [kind] * n}
        human_like = cohort in ("human", "screen_reader", "legitimate_agent")

        # ── context ──
        if cohort == "legitimate_agent":
            d["actor"] = ["assistant"] * n
            d["agent_token"] = pick(n, ["valid", "none"], [0.95, 0.05])
            d["amount_eur"] = lognorm(n, 30, 0.6)
        elif cohort == "bot":
            imp = {"script": 0.20, "a11y_agent": 0.0, "farm": 0.10}[kind]
            d["actor"] = np.where(rng.random(n) < imp, "assistant", "user")
            tok = pick(n, ["forged", "none"], [0.4, 0.6])
            d["agent_token"] = np.where(d["actor"] == "assistant", tok, "none")
            d["amount_eur"] = lognorm(n, 180, 0.9)
        else:
            d["actor"] = ["user"] * n
            d["agent_token"] = ["none"] * n
            d["amount_eur"] = lognorm(n, 60, 1.0)
        d["new_payee"] = flag(n, {"human": 0.12, "screen_reader": 0.12,
                                  "legitimate_agent": 0.08, "bot": 0.7}[cohort])

        # ── layer 1: device integrity (Play Integrity verdict) ──
        if human_like:
            d["integrity"] = pick(n, ["device_ok", "basic", "no_label", "missing"], [0.97, 0.017, 0.008, 0.005])
            d["app_recognized"] = flag(n, 0.997)
            d["device_activity"] = pick(n, [1, 2, 3, 4], [0.85, 0.14, 0.01, 0.0])
        elif kind == "script":
            d["integrity"] = pick(n, ["device_ok", "basic", "no_label", "missing"], [0.10, 0.20, 0.55, 0.15])
            d["app_recognized"] = flag(n, 0.80)
            d["device_activity"] = pick(n, [1, 2, 3, 4], [0.4, 0.3, 0.2, 0.1])
        elif kind == "a11y_agent":
            d["integrity"] = pick(n, ["device_ok", "basic", "no_label", "missing"], [0.78, 0.12, 0.07, 0.03])
            d["app_recognized"] = flag(n, 0.98)
            d["device_activity"] = pick(n, [1, 2, 3, 4], [0.6, 0.3, 0.08, 0.02])
        else:  # farm: real, clean phones on a rack
            d["integrity"] = pick(n, ["device_ok", "basic", "no_label", "missing"], [0.85, 0.08, 0.05, 0.02])
            d["app_recognized"] = flag(n, 0.98)
            d["device_activity"] = pick(n, [1, 2, 3, 4], [0.15, 0.25, 0.35, 0.25])

        # accessibility: any service on, and whether Play says an app from outside Play can control the device
        a11y_p = {"human": 0.06, "screen_reader": 1.0, "legitimate_agent": 0.06,
                  "script": 0.10, "a11y_agent": 1.0, "farm": 0.30}[kind or cohort]
        d["accessibility_service_on"] = flag(n, a11y_p)
        unk_p = {"human": 0.10, "screen_reader": 0.06, "legitimate_agent": 0.10,
                 "script": 0.5, "a11y_agent": 0.45, "farm": 0.5}[kind or cohort]
        d["unknown_controlling_app"] = d["accessibility_service_on"] * flag(n, unk_p)

        # ── layer 2: RASP ──
        rasp_p = {  # rooted, emulator, hooked, debugger, signature_invalid
            "human": (0.015, 0.002, 0.001, 0.0005, 0.001),
            "screen_reader": (0.015, 0.002, 0.001, 0.0005, 0.001),
            "legitimate_agent": (0.015, 0.002, 0.001, 0.0005, 0.001),
            "script": (0.50, 0.60, 0.45, 0.05, 0.15),
            "a11y_agent": (0.08, 0.02, 0.03, 0.0, 0.02),
            "farm": (0.15, 0.03, 0.05, 0.0, 0.02),
        }[kind or cohort]
        for col, p in zip(["rasp_rooted", "rasp_emulator", "rasp_hooked", "rasp_debugger"], rasp_p[:4]):
            d[col] = flag(n, p)
        d["rasp_signature_ok"] = 1 - flag(n, rasp_p[4])

        # ── layer 3: behavior (derived features only) ──
        a11y = d["accessibility_service_on"]
        if cohort == "human":
            d["a11y_action_ratio"] = np.where(a11y == 1, beta(n, 1, 25), 0.0)
            d["touch_interval_cv"] = lognorm(n, 0.60, 0.35)
            pasted = flag(n, 0.05)
            d["keystroke_interval_cv"] = np.where(pasted == 1, 0.0, lognorm(n, 0.45, 0.35))
            d["swipe_linearity"] = beta(n, 9, 3)
            d["time_to_confirm_s"] = lognorm(n, 7.0, 0.6)
            d["screens_before_transfer"] = 2 + rng.poisson(1.2, n)
            d["nav_path"] = pick(n, ["home>send>confirm", "home>contacts>send>confirm", "notification>confirm"],
                                 [0.55, 0.30, 0.15])
            on_desk = flag(n, 0.12)
            d["motion_energy"] = np.where(on_desk == 1, lognorm(n, 0.06, 0.6), lognorm(n, 0.45, 0.5))
        elif cohort == "screen_reader":
            d["a11y_action_ratio"] = beta(n, 18, 2)
            d["touch_interval_cv"] = lognorm(n, 0.24, 0.40)
            dictation = flag(n, 0.25)  # voice or braille input arrives as injected text
            d["keystroke_interval_cv"] = np.where(dictation == 1, 0.0, lognorm(n, 0.28, 0.4))
            d["swipe_linearity"] = beta(n, 30, 2)
            d["time_to_confirm_s"] = lognorm(n, 16.0, 0.6)
            d["screens_before_transfer"] = 3 + rng.poisson(2.0, n)
            d["nav_path"] = pick(n, ["home>send>confirm", "home>contacts>send>confirm", "notification>confirm"],
                                 [0.50, 0.40, 0.10])
            on_desk = flag(n, 0.25)
            d["motion_energy"] = np.where(on_desk == 1, lognorm(n, 0.06, 0.6), lognorm(n, 0.40, 0.5))
        elif cohort == "legitimate_agent":
            d["a11y_action_ratio"] = np.where(a11y == 1, beta(n, 1, 25), 0.0)
            d["touch_interval_cv"] = lognorm(n, 0.50, 0.40)
            d["keystroke_interval_cv"] = lognorm(n, 0.45, 0.35)  # the user typed the chat prompt
            d["swipe_linearity"] = beta(n, 9, 3)
            d["time_to_confirm_s"] = lognorm(n, 1.6, 0.5)  # the assistant pre-fills; one tap
            d["screens_before_transfer"] = np.ones(n, dtype=int)
            d["nav_path"] = ["chat>confirm"] * n
            on_desk = flag(n, 0.12)
            d["motion_energy"] = np.where(on_desk == 1, lognorm(n, 0.06, 0.6), lognorm(n, 0.45, 0.5))
        elif kind == "script":
            d["a11y_action_ratio"] = np.where(a11y == 1, beta(n, 20, 1.5), 0.0)
            d["touch_interval_cv"] = lognorm(n, 0.05, 0.6)
            d["keystroke_interval_cv"] = np.where(flag(n, 0.7) == 1, 0.0, lognorm(n, 0.04, 0.6))
            d["swipe_linearity"] = beta(n, 60, 1)
            d["time_to_confirm_s"] = lognorm(n, 0.6, 0.6)
            d["screens_before_transfer"] = np.ones(n, dtype=int)
            d["nav_path"] = pick(n, ["deeplink>confirm", "home>send>confirm"], [0.8, 0.2])
            d["motion_energy"] = lognorm(n, 0.01, 0.8)
        elif kind == "a11y_agent":  # LLM-driven malicious app working the UI through accessibility
            d["a11y_action_ratio"] = beta(n, 25, 1.5)
            d["touch_interval_cv"] = lognorm(n, 0.20, 0.45)  # LLM latency adds jitter
            d["keystroke_interval_cv"] = np.where(flag(n, 0.85) == 1, 0.0, lognorm(n, 0.15, 0.5))
            d["swipe_linearity"] = beta(n, 30, 1.5)
            d["time_to_confirm_s"] = lognorm(n, 4.5, 0.6)
            d["screens_before_transfer"] = 2 + rng.poisson(1.0, n)
            d["nav_path"] = pick(n, ["home>send>confirm", "home>contacts>send>confirm"], [0.7, 0.3])
            in_pocket = flag(n, 0.4)
            d["motion_energy"] = np.where(in_pocket == 1, lognorm(n, 0.30, 0.6), lognorm(n, 0.05, 0.7))
        else:  # farm: adb/HID taps with added jitter on racked phones
            d["a11y_action_ratio"] = np.where(a11y == 1, beta(n, 15, 2), beta(n, 1, 40))
            d["touch_interval_cv"] = lognorm(n, 0.35, 0.45)
            d["keystroke_interval_cv"] = lognorm(n, 0.20, 0.5)
            d["swipe_linearity"] = beta(n, 20, 2)
            d["time_to_confirm_s"] = lognorm(n, 2.5, 0.6)
            d["screens_before_transfer"] = 1 + rng.poisson(0.8, n)
            d["nav_path"] = pick(n, ["deeplink>confirm", "home>send>confirm"], [0.5, 0.5])
            d["motion_energy"] = lognorm(n, 0.02, 0.7)
        return pd.DataFrame(d)

    parts = [block(c, n) for c, n in COUNTS.items() if c != "bot"]
    parts += [block("bot", n, k) for k, n in BOT_KINDS.items()]
    df = pd.concat(parts, ignore_index=True)
    df = df.iloc[rng.permutation(len(df))].reset_index(drop=True)
    df["bot_kind"] = df["bot_kind"].replace("", "none")

    df.insert(0, "session_id", [f"s{i:05d}" for i in range(1, len(df) + 1)])
    df.insert(1, "label", df["cohort"].map({"human": "human", "screen_reader": "human",
                                            "legitimate_agent": "legitimate_agent", "bot": "bot"}))
    df["amount_eur"] = df["amount_eur"].clip(1, 5000).round(2)
    for col in ["a11y_action_ratio", "touch_interval_cv", "keystroke_interval_cv", "swipe_linearity", "motion_energy"]:
        df[col] = df[col].clip(0, 1.5 if "cv" in col else 1).round(3)
    df["time_to_confirm_s"] = df["time_to_confirm_s"].clip(0.1, 120).round(2)
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str)
    int_cols = ["new_payee", "app_recognized", "device_activity", "accessibility_service_on",
                "unknown_controlling_app", "rasp_rooted", "rasp_emulator", "rasp_hooked", "rasp_debugger",
                "rasp_signature_ok", "screens_before_transfer"]
    df[int_cols] = df[int_cols].astype(int)
    return df


def fingerprint(df: pd.DataFrame) -> str:
    return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()[:12]

In [ ]:
# @title Load the sessions
from pathlib import Path

EXPECTED_FINGERPRINT = "251dd26c13fb"

if Path("sessions.csv").exists():            # running from a clone of the repo
    df = pd.read_csv("sessions.csv", keep_default_na=False)
    source = "sessions.csv"
else:                                       # Colab: build it here, same seed, same rows
    df = make_sessions()
    source = "generated in this notebook"

fp = fingerprint(df)
print(f"{len(df):,} sessions, {source}. Fingerprint {fp}", "✓" if fp == EXPECTED_FINGERPRINT else "✗ (differs from the repo copy)")
df.groupby(["label", "cohort"]).size().rename("sessions").to_frame()

### What's in a row

| Layer | Columns | Where it comes from |
|---|---|---|
| Integrity | `integrity` (device_ok · basic · no_label · missing), `app_recognized`, `device_activity` (1–4), `unknown_controlling_app` | Play Integrity verdict, decoded on the server |
| RASP | `rasp_rooted`, `rasp_emulator`, `rasp_hooked`, `rasp_debugger`, `rasp_signature_ok` | the app's runtime checks (`X-RASP`) |
| Agent | `actor` (user · assistant), `agent_token` (none · valid · forged) | the agent-identity layer |
| Context | `amount_eur`, `new_payee` | the transfer itself |
| **Behavior** | `touch_interval_cv` (how uneven the taps are), `swipe_linearity` (1 = ruler-straight), `keystroke_interval_cv` (0 = text arrived without keystrokes), `time_to_confirm_s`, `motion_energy` (phone in a hand vs on a desk or rack), `a11y_action_ratio` (share of actions delivered through an accessibility service), `accessibility_service_on`, `screens_before_transfer`, `nav_path` | derived on the device; never what was typed |

`cohort` splits the humans into **human** and **screen_reader**: real people using a screen reader, whose interaction runs through an accessibility service. `bot_kind` says what kind of automation a bot is (script, a11y_agent, farm). Neither is a feature: it's only for evaluation.

In [ ]:
df.groupby("label")[["touch_interval_cv", "swipe_linearity", "keystroke_interval_cv",
                      "time_to_confirm_s", "motion_energy", "a11y_action_ratio"]].median().round(2)

## 0 · The baseline: today's server

These are the weights and bands from `server/risk.py` (and the layer files next to it). Each layer adds its **worst** signal's points, the total is capped at 100, and the band gives the decision.
Some signals also set a **floor**: a hard rule such as “a forged agent token is always blocked” or “no integrity verdict always steps up”. Floors stay as they are in this lab; you tune the score and the bands.

In [ ]:
# Copied from server/: risk.py (bands), verify.py (integrity), rasp.py, agent_tokens.py, config.py (limits)
BANDS = (25, 50, 90)                 # allow < 25 ≤ monitor < 50 ≤ step_up < 90 ≤ block
DECISIONS = np.array(["allow", "monitor", "step_up", "block"])
ALLOW, MONITOR, STEP_UP, BLOCK = range(4)

AGENT_LIMIT_EUR, USER_LIMIT_EUR = 100, 1000

def server_layers(df):
    """Points per layer (worst signal wins) and the strictest floor, as the server computes them."""
    s = pd.DataFrame(index=df.index)
    s["integrity"] = np.maximum.reduce([
        df["integrity"].map({"device_ok": 0, "basic": 20, "no_label": 35, "missing": 30}).to_numpy(),
        np.where(df["app_recognized"] == 0, 30, 0),
        np.where(df["unknown_controlling_app"] == 1, 30, 0),
        df["device_activity"].map({1: 0, 2: 0, 3: 10, 4: 20}).to_numpy(),
    ])
    s["rasp"] = np.maximum.reduce([
        np.where(df["rasp_hooked"] == 1, 40, 0),
        np.where(df["rasp_signature_ok"] == 0, 40, 0),
        np.where(df["rasp_debugger"] == 1, 30, 0),
        np.where(df["rasp_rooted"] == 1, 30, 0),
        np.where(df["rasp_emulator"] == 1, 25, 0),
    ])
    assistant = df["actor"] == "assistant"
    token = df["agent_token"]
    out_of_scope = assistant & (token == "valid") & (df["amount_eur"] > AGENT_LIMIT_EUR)
    s["agent"] = np.select(
        [assistant & (token == "forged"), assistant & (token == "none"), out_of_scope, assistant],
        [40, 25, 30, 10], 0)
    step_up_floor = (df["integrity"].isin(["no_label", "missing"]) | (df["app_recognized"] == 0)
                     | (df["unknown_controlling_app"] == 1) | (df["rasp_hooked"] == 1)
                     | (df["rasp_signature_ok"] == 0) | (df["rasp_debugger"] == 1)
                     | (assistant & (token == "none")) | out_of_scope
                     | (~assistant & (df["amount_eur"] > USER_LIMIT_EUR)))
    s["floor"] = np.select([assistant & (token == "forged"), step_up_floor], [BLOCK, STEP_UP], ALLOW)
    s["points"] = s[["integrity", "rasp", "agent"]].sum(axis=1).clip(upper=100)
    return s

def decide(points, floor, bands):
    """Band the score, then raise it to the floor. Returns 0..3 (allow..block)."""
    p = np.minimum(np.asarray(points), 100)
    d = np.select([p < bands[0], p < bands[1], p < bands[2]], [ALLOW, MONITOR, STEP_UP], BLOCK)
    return np.maximum(d, np.asarray(floor))

server = server_layers(df)
baseline = decide(server["points"], server["floor"], BANDS)
pd.Series(DECISIONS[baseline]).value_counts().reindex(DECISIONS).rename("sessions").to_frame().T

In [ ]:
# @title Helpers: the numbers we watch
def challenged(decisions):
    return pd.Series(np.asarray(decisions) >= STEP_UP, index=df.index)

def three_numbers(decisions, title=""):
    """Slide 25's three numbers."""
    ch = challenged(decisions)
    nums = {
        "Bots caught": ch[df["label"] == "bot"].mean(),
        "Humans challenged": ch[df["label"] == "human"].mean(),
        "Agents misread as bots": ch[df["label"] == "legitimate_agent"].mean(),
    }
    print(title)
    for k, v in nums.items():
        print(f"  {k:<24}{v:6.1%}")
    return nums

def cohort_report(decisions):
    """Who gets challenged (step_up or block) and who gets blocked, per cohort."""
    d = pd.Series(np.asarray(decisions), index=df.index)
    groups = df["cohort"].where(df["cohort"] != "bot", "bot · " + df["bot_kind"])
    t = pd.DataFrame({"sessions": groups.value_counts(),
                      "challenged": (d >= STEP_UP).groupby(groups).mean(),
                      "blocked": (d == BLOCK).groupby(groups).mean()})
    order = ["human", "screen_reader", "legitimate_agent", "bot · script", "bot · a11y_agent", "bot · farm"]
    t = t.reindex(order)
    t[["challenged", "blocked"]] = t[["challenged", "blocked"]].map("{:.1%}".format)
    return t

three_numbers(baseline, "Baseline (server as shipped, no behavior layer):");

The server already stops the crude bots (rooted emulators, hooked apps, forged agent tokens).
But look at what's left: automation on **real, clean phones** passes every device check. That's what the behavior layer is for.

## 1 · Build the score  *(slide 25)*

Below are six behavioral **tells**: yes/no facts about a session that make it look automated.
Give each one some points (0–30). The behavior layer adds their sum, capped at 60, on top of the server's layers.

Start simple. A weighted sum is plenty: the lesson is about thresholds and trade-offs, not model accuracy.

In [ ]:
# @title The tells (run this)
def tells(df):
    return pd.DataFrame({
        "regular_taps":     df["touch_interval_cv"] < 0.30,      # metronome-like tapping
        "straight_swipes":  df["swipe_linearity"] > 0.90,        # ruler-straight swipes
        "fast_confirm":     df["time_to_confirm_s"] < 3,         # confirmed faster than a person reads
        "a11y_driven":      df["a11y_action_ratio"] > 0.50,      # UI driven through an accessibility service
        "phone_not_moving": df["motion_energy"] < 0.10,          # no hand holding the phone
        "injected_text":    df["keystroke_interval_cv"] < 0.05,  # text arrived without keystrokes
    })

T = tells(df)
(T.groupby(df["label"]).mean() * 100).round(0).astype(int).add_suffix(" %")

In [ ]:
# ✏️ TODO 1 — points for each tell (0–30). All zeros = no behavior layer yet.
WEIGHTS = {
    "regular_taps":     0,
    "straight_swipes":  0,
    "fast_confirm":     0,
    "a11y_driven":      0,
    "phone_not_moving": 0,
    "injected_text":    0,
}
BEHAVIOR_CAP = 60

In [ ]:
def behavior_points(df):
    t = tells(df)
    return sum(w * t[name] for name, w in WEIGHTS.items()).clip(upper=BEHAVIOR_CAP).astype(int)

behavior = behavior_points(df)
total = (server["points"] + behavior).clip(upper=100)
if behavior.max() == 0:
    print("Your behavior layer adds nothing yet: give some tells points in TODO 1.")
scored = decide(total, server["floor"], BANDS)
three_numbers(baseline, "Baseline:")
three_numbers(scored, f"With your behavior layer, bands {BANDS}:");

## 2 · Choose the thresholds  *(slide 25)*

The chart sweeps the **step-up** threshold for your current score: everything at or above it is challenged.
Moving it left catches more bots and challenges more people. Pick bands for allow / monitor / step_up / block.

In [ ]:
# @title Trade-off chart
GROUPS_STEP1 = {  # slide 25's three numbers
    "Bots caught": ("#2a78d6", df["label"] == "bot"),
    "Humans challenged": ("#eb6834", df["label"] == "human"),
    "Agents misread as bots": ("#6250d6", df["label"] == "legitimate_agent"),
}

def tradeoff_chart(points, floor, title, mark=None, groups=GROUPS_STEP1):
    xs = np.arange(5, 101, 5)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for name, (color, mask) in groups.items():
        m = mask.to_numpy()
        ys = [100 * (decide(points, floor, (min(x, 25), x, 101))[m] >= STEP_UP).mean() for x in xs]
        ax.plot(xs, ys, color=color, lw=2, label=name)
    if mark is not None:
        ax.axvline(mark, color="#8a8984", lw=1, ls="--")
        ax.annotate(f"step-up at {mark}", (mark, 101), xytext=(4, -12), textcoords="offset points",
                    fontsize=9, color="#52514e")
    ax.set(xlabel="step-up threshold (score)", ylabel="% challenged (step-up or block)",
           ylim=(0, 102), xlim=(5, 100), title=title)
    ax.grid(axis="y", color="#e6e5e0", lw=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    plt.show()

tradeoff_chart(total, server["floor"], "Your score: who gets challenged at each step-up threshold", mark=BANDS[1])

In [ ]:
# ✏️ TODO 2 — your bands: allow < a ≤ monitor < b ≤ step_up < c ≤ block
MY_BANDS = (25, 50, 90)   # the server's defaults

decisions = decide(total, server["floor"], MY_BANDS)
three_numbers(decisions, f"Your score, bands {MY_BANDS}:")
print("\n✍️  Write these three numbers down: slide 26 asks you to look again.")

## 3 · Now look at who you're challenging  *(slide 26)*

“Humans challenged” is an average over everyone. Split it by cohort.

In [ ]:
# ✏️ TODO 3 — run the per-cohort report on your decisions (replace None with `decisions`)
report_for = None
cohort_report(report_for) if report_for is not None else print("Set report_for = decisions")

Screen-reader users interact *through* an accessibility service: actions arrive as accessibility events, swipes are clean scroll actions, and taps follow the screen reader's rhythm.
A score built on “typical” users reads all of that as automation. (Slide 5: accessibility services are both a malware tool and a lifeline.)

**What actually separates them?** Compare the screen-reader cohort with the bots that drive the UI through accessibility:

In [ ]:
a11y_sessions = df["a11y_action_ratio"] > 0.5
cols = ["time_to_confirm_s", "keystroke_interval_cv", "motion_energy", "screens_before_transfer"]
df[a11y_sessions].groupby("cohort")[cols].median().round(2)

## 4 · Fix it  *(slide 26)*

Three fixes, the same ones you'd use for any fairness problem in ML. They're already written: **switch them on** in TODO 4 and watch the per-cohort table change.

1. **Separate features.** Raw-touch tells (`regular_taps`, `straight_swipes`) measure the screen reader, not the person, when actions arrive through accessibility: only count them on raw-touch sessions. Being accessibility-driven isn't a tell on its own; *how* the session behaves is (speed, injected text, a phone that never moves). And a declared assistant with a valid agent token is *supposed* to be fast.
2. **Never block on behavior alone.** If the device, RASP and agent layers wouldn't challenge a session, behavior can raise it to step-up at most. Step-up is a fingerprint tap; a block is a lost customer.
3. **Evaluate per cohort**, then re-tune the bands.

In [ ]:
# ✏️ TODO 4 — switch the fixes on (True), one at a time, and re-run this cell and the two below.
FIX_SEPARATE_FEATURES      = False   # 1: raw-touch tells only count on raw-touch sessions; being
                                     #    accessibility-driven isn't a tell, being fast/injected is
FIX_DECLARED_AGENT_IS_FAST = False   # 1: a declared assistant with a valid agent token may be fast
NO_BLOCK_ON_BEHAVIOR_ALONE = False   # 2: behavior can raise a session to step-up, never to block
MY_BANDS_V2 = MY_BANDS               # 3: re-tune after the fixes, e.g. (25, 45, 90)

In [ ]:
# The fixed behavior layer, driven by your switches and your TODO 1 weights. Read it; no need to edit.
def behavior_points_v2(df):
    t = tells(df)
    a11y = t["a11y_driven"]
    declared = (df["actor"] == "assistant") & (df["agent_token"] == "valid")
    W = WEIGHTS
    if not FIX_SEPARATE_FEATURES:
        if FIX_DECLARED_AGENT_IS_FAST:
            t = t.assign(fast_confirm=t["fast_confirm"] & ~declared)
        points = sum(w * t[name] for name, w in W.items())
        return pd.Series(points, index=df.index).clip(upper=BEHAVIOR_CAP).astype(int)

    raw = ~a11y
    # on accessibility-driven sessions, "fast" means faster than anyone listening to a screen reader (8 s)
    fast = df["time_to_confirm_s"] < np.where(a11y, 8, 3)
    if FIX_DECLARED_AGENT_IS_FAST:
        fast = fast & ~declared
    # a11y-driven isn't a tell by itself: its points move onto speed, the thing that separates
    # a bot from a person listening to a screen reader; injected text weighs a bit more there too
    points = (W["regular_taps"] * (t["regular_taps"] & raw)
              + W["straight_swipes"] * (t["straight_swipes"] & raw)
              + np.where(a11y, W["a11y_driven"], W["fast_confirm"]) * fast
              + W["phone_not_moving"] * t["phone_not_moving"]
              + np.where(a11y, W["injected_text"] + 5, W["injected_text"]) * t["injected_text"])
    return pd.Series(points, index=df.index).clip(upper=BEHAVIOR_CAP).astype(int)

In [ ]:
behavior_v2 = behavior_points_v2(df)
total_v2 = (server["points"] + behavior_v2).clip(upper=100)
decisions_v2 = decide(total_v2, server["floor"], MY_BANDS_V2)
if NO_BLOCK_ON_BEHAVIOR_ALONE:
    without_behavior = decide(server["points"], server["floor"], MY_BANDS_V2)
    decisions_v2 = np.where((decisions_v2 == BLOCK) & (without_behavior < STEP_UP), STEP_UP, decisions_v2)

three_numbers(decisions, f"Before the fix, bands {MY_BANDS}:")
three_numbers(decisions_v2, f"After the fix, bands {MY_BANDS_V2}:")
cohort_report(decisions_v2)

In [ ]:
GROUPS_STEP2 = {
    "Bots caught": ("#2a78d6", df["cohort"] == "bot"),
    "Humans (typical)": ("#eb6834", df["cohort"] == "human"),
    "Screen-reader users": ("#1baf7a", df["cohort"] == "screen_reader"),
    "Legitimate agents": ("#6250d6", df["cohort"] == "legitimate_agent"),
}
tradeoff_chart(total, server["floor"], "Before the fix, per cohort", mark=MY_BANDS[1], groups=GROUPS_STEP2)
tradeoff_chart(total_v2, server["floor"], "After the fix, per cohort", mark=MY_BANDS_V2[1], groups=GROUPS_STEP2)

### Slide 26 numbers

The deck shows three numbers. *Before* is your step-1 score at your bands; *after* is the fixed score.

In [ ]:
def slide_26(decisions):
    ch = challenged(decisions)
    return {"Bots caught": ch[df["label"] == "bot"].mean(),
            "All humans challenged": ch[df["label"] == "human"].mean(),
            "Screen-reader users challenged": ch[df["cohort"] == "screen_reader"].mean(),
            "Screen-reader users blocked": (np.asarray(decisions) == BLOCK)[(df["cohort"] == "screen_reader").to_numpy()].mean()}

pd.DataFrame({"baseline (server)": slide_26(baseline),
              f"before: step 1, bands {MY_BANDS}": slide_26(decisions),
              f"after: fixed, bands {MY_BANDS_V2}": slide_26(decisions_v2)}).map("{:.1%}".format)

**What's left.** Screen-reader users are still challenged more often than other people. Most of that isn't behavior at all: some use screen readers installed from outside Google Play, and the integrity layer's hard rule (“an unknown app can control the device → step-up”) catches them. That's the per-cohort lesson: every layer needs this check, not just the ML one.

### Optional · ML track (2 min)

Prefer a model? This trains a logistic regression on the behavioral columns (bot vs. everyone else) and plugs its probability in as the behavior layer.
Compare its screen-reader row with the typical-human row. Only 5% of the training data is screen-reader users, so the model has little reason to get them right: whatever gap is left, you only see it because you split by cohort.

In [ ]:
# @title Logistic regression as the behavior layer (optional; scikit-learn is preinstalled in Colab)
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
except ImportError:
    print("scikit-learn isn't installed here; skip this cell (it's preinstalled in Colab).")
else:
    features = ["touch_interval_cv", "swipe_linearity", "keystroke_interval_cv", "time_to_confirm_s",
                "motion_energy", "a11y_action_ratio", "screens_before_transfer"]
    X = df[features].assign(time_to_confirm_s=np.log(df["time_to_confirm_s"]))
    y = (df["label"] == "bot").astype(int)
    train, test = train_test_split(df.index, test_size=0.5, random_state=0, stratify=df["cohort"])
    model = LogisticRegression(max_iter=1000).fit(X.loc[train], y.loc[train])
    ml_points = pd.Series(np.round(model.predict_proba(X)[:, 1] * BEHAVIOR_CAP), index=df.index)
    ml_decisions = decide((server["points"] + ml_points).clip(upper=100), server["floor"], MY_BANDS)
    test_mask = df.index.isin(test)
    print(f"Logistic regression as the behavior layer, bands {MY_BANDS}, held-out half only:")
    for cohort in ["bot", "human", "screen_reader", "legitimate_agent"]:
        m = test_mask & (df["cohort"] == cohort).to_numpy()
        print(f"  {cohort:<18}{(ml_decisions[m] >= STEP_UP).mean():6.1%} challenged")

## Take-aways

* Device and runtime evidence stop crude automation; **behavior** is what catches automation on clean, real phones.
* An overall false-positive rate can look fine while one group of real people is challenged most of the time. **Evaluate per cohort.**
* Separate features that describe *the assistive technology* from features that describe *the session*.
* **Never hard-block on behavior alone.** Route uncertainty to step-up, and ship new checks in monitor mode first (slide 27).